<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.2. Auto MPG Regression - Neural Networks
  </strong>
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from rich.table import Table
from rich.console import Console

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    Normalizer,
    PolynomialFeatures,
)
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

## Data Loading

In [ ]:
df = sns.load_dataset("mpg")

X = df.drop(columns=["mpg"])
y = df["mpg"].values

## Clean data

In [ ]:
X["origin"] = pd.Categorical(X["origin"]).codes
X["horsepower"] = X["horsepower"].fillna(X["horsepower"].median())
X = X.drop(columns=["name"])

## Preprocessing of data

Split the data with all features into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

console.print(f"Training set size: {X_train.shape}")
console.print(f"Test set size: {X_test.shape}")

Method to escalate the data

In [ ]:
def scaler_data(scaler, X_train, y_train, X_test, y_test):

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1))
    y_test_scaled = scaler.transform(y_test.reshape(-1, 1))

    return X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled

Method to convert to tensors

In [ ]:
def to_tensors(X_train, y_train, X_test, y_test):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

    return X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor

### Configurations

Scalers and arhiquitecture to compare in a neural network.

In [ ]:
scalers = {
    "Min-Max": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Mean Normalization": Normalizer(),
}

architectures = {
    "1-Layer (32)": [32],
    "1-Layer (64)": [64],
    "2-Layers (64-32)": [64, 32],
    "2-Layers (128-64)": [128, 64],
    "3-Layers (128-64-32)": [128, 64, 32],
    "3-Layers (256-128-64)": [256, 128, 64],
}

## Model Architecture

In [ ]:
class RegressionNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size=1):
        super(RegressionNetwork, self).__init__()

        layers = []
        last_size = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(last_size, h))
            layers.append(nn.ReLU())
            last_size = h

        layers.append(nn.Linear(last_size, output_size))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Training Methods

Method to train a regression model

In [ ]:
def train_linear_model(
    X_train_scaled, y_train_scaled, X_test_scaled, model_type, degree=1
):

    if model_type == "ridge":
        model = Ridge(alpha=1.0)
    else:
        poly = PolynomialFeatures(degree=degree)
        X_train_scaled = poly.fit_transform(X_train_scaled)
        X_test_scaled = poly.transform(X_test_scaled)
        model = LinearRegression()

    model.fit(X_train_scaled, y_train_scaled)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return y_train_pred, y_test_pred

Method to train a neural network model

In [ ]:
def train_nn_model(model, X_train_tensor, y_train_tensor, epochs, lr):
    loss_history = []

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        y_pred = model(X_train_tensor)
        loss = criterion(y_pred, y_train_tensor)

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    return model, loss_history

### Evaluation Methods

Method to calculate metrics

In [ ]:
def calculate_regression_metrics(y_test, y_test_pred):
    mse = mean_squared_error(y_test, y_test_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)

    return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2}


def calculate_all_metrics(y_train, y_train_pred, y_test, y_test_pred):
    train_metrics = calculate_regression_metrics(y_train, y_train_pred)
    test_metrics = calculate_regression_metrics(y_test, y_test_pred)

    return {
        "train_mse": train_metrics["mse"],
        "train_rmse": train_metrics["rmse"],
        "train_mae": train_metrics["mae"],
        "train_r2": train_metrics["r2"],
        "test_mse": test_metrics["mse"],
        "test_rmse": test_metrics["rmse"],
        "test_mae": test_metrics["mae"],
        "test_r2": test_metrics["r2"],
    }

Method to evaluate a neural network model

In [ ]:
def evaluate_nn_model(model, X_train_tensor, X_test_tensor):
    model.eval()
    with torch.no_grad():
        y_train_pred = model(X_train_tensor).numpy()
        y_test_pred = model(X_test_tensor).numpy()

    return y_train_pred.flatten(), y_test_pred.flatten()  ## ver

Method to train and evaluate network model

In [ ]:
def train_and_evaluate_nn(
    hidden_sizes,
    X_train_tensor,
    y_train_tensor,
    X_test_tensor,
    y_train,
    y_test,
    epochs,
    lr,
):
    torch.manual_seed(42)

    model = RegressionNetwork(X_train_tensor.shape[1], hidden_sizes)
    model, loss_history = train_nn_model(
        model, X_train_tensor, y_train_tensor, epochs, lr
    )

    y_train_pred, y_test_pred = evaluate_nn_model(model, X_train_tensor, X_test_tensor)
    metrics = calculate_all_metrics(y_train, y_train_pred, y_test, y_test_pred)

    return metrics, loss_history

## Visualization methods

Method to create the comparison table

In [ ]:
def create_regression_comparison_table(title, results):
    table = Table(title=title)

    table.add_column("Normalization", justify="left", style="cyan")
    table.add_column("Model", justify="center", style="green", width=20)
    table.add_column("Train\nMSE", justify="right", style="green")
    table.add_column("Test\nMSE", justify="right", style="green")
    table.add_column("Train\nRMSE", justify="right", style="green")
    table.add_column("Test\nRMSE", justify="right", style="green")
    table.add_column("Train\nMAE", justify="right", style="green")
    table.add_column("Test\nMAE", justify="right", style="green")
    table.add_column("Train\nR²", justify="right", style="green")
    table.add_column("Test\nR²", justify="right", style="green")

    for scaler_name, scaler_metrics in results.items():
        for model_name, metrics in scaler_metrics.items():
            table.add_row(
                scaler_name,
                model_name,
                f'{metrics["train_mse"]:.4f}',
                f'{metrics["test_mse"]:.4f}',
                f'{metrics["train_rmse"]:.4f}',
                f'{metrics["test_rmse"]:.4f}',
                f'{metrics["train_mae"]:.4f}',
                f'{metrics["test_mae"]:.4f}',
                f'{metrics["train_r2"]:.4f}',
                f'{metrics["test_r2"]:.4f}',
            )

    console.print(table)

In [ ]:
def plot_loss_curves(loss_curves, scalers, architectures):
    n_cols = len(architectures)
    n_rows = len(scalers)

    plt.figure(figsize=(20, 12))

    for i, (label, history) in enumerate(loss_curves.items(), 1):
        plt.subplot(n_rows, n_cols, i)
        plt.plot(history, label="Training Loss")
        plt.title(label, fontsize=10)
        plt.xlabel("Epochs")
        plt.ylabel("MSE Loss")
        plt.grid(True)
        plt.yscale("log")

    plt.tight_layout()
    plt.show()

## 1. Create a neural network with different network architectures.

In [ ]:
learning_rate = 0.001
epochs = 500

scaler_metrics = {}
loss_curves = {}

for scaler_name, scaler in scalers.items():

    X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled = scaler_data(
        scaler, X_train, y_train, X_test, y_test
    )

    X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor = to_tensors(
        X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled
    )
    scaler_metrics[scaler_name] = {}

    for arch_name, hidden_sizes in architectures.items():
        metrics, loss_history = train_and_evaluate_nn(
            hidden_sizes,
            X_train_tensor,
            y_train_tensor,
            X_test_tensor,
            y_train_scaled,
            y_test_scaled,
            epochs,
            learning_rate,
        )

        scaler_metrics[scaler_name][arch_name] = metrics
        loss_curves[f"{scaler_name} | {arch_name}"] = loss_history

create_regression_comparison_table(
    "Neural Network Performance Comparison - Different Architectures and Scalers",
    scaler_metrics,
)

It might seem that the Min-Max (64-32) model is the best option due to its slightly higher R² compared to the other metrics. However, when analyzing the other metrics, such as MSE, RMSE, and MAE, we can see that the `two-layer Min-Max model (128-64)` outperforms it, as it achieves significantly lower error values and maintains a good R². With the above, we could say that this model not only learns patterns effectively, but also generalizes better on unseen data. 

## 2. Compare performance with previous regression models

In [ ]:
best_scaler = {"Min-Max": MinMaxScaler()}
scaler_name, scaler_obj = list(best_scaler.items())[0]

X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled = scaler_data(
    scaler_obj, X_train, y_train, X_test, y_test
)

comparison_metrics = {}
comparison_metrics[scaler_name] = {}

Train a linear model using ridge

In [ ]:
y_train_pred_ridge, y_test_pred_ridge = train_linear_model(
    X_train_scaled, y_train_scaled, X_test_scaled, "ridge"
)
metrics_ridge = calculate_all_metrics(
    y_train_scaled, y_train_pred_ridge, y_test_scaled, y_test_pred_ridge
)
comparison_metrics[scaler_name]["Ridge Regression"] = metrics_ridge

Train a linear regression model

In [ ]:
y_train_pred_linear, y_test_pred_linear = train_linear_model(
    X_train_scaled, y_train_scaled, X_test_scaled, "linear"
)
metrics_linear = calculate_all_metrics(
    y_train_scaled, y_train_pred_linear, y_test_scaled, y_test_pred_linear
)
comparison_metrics[scaler_name]["Linear Regression"] = metrics_linear

Train a polynomial regression model with the best degree (2) of previous laboratories

In [ ]:
y_train_pred_linear, y_test_pred_linear = train_linear_model(
    X_train_scaled, y_train_scaled, X_test_scaled, model_type="poly", degree=2
)

metrics_linear = calculate_all_metrics(
    y_train_scaled, y_train_pred_linear, y_test_scaled, y_test_pred_linear
)
comparison_metrics[scaler_name]["Polynomial Regression"] = metrics_linear

Train best neural network (based on previous results)

In [ ]:
X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor = to_tensors(
    X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled
)

torch.manual_seed(42)
best_architecture = [128, 64]
best_model = RegressionNetwork(X_train_tensor.shape[1], best_architecture)
best_model, best_loss_history = train_nn_model(
    best_model, X_train_tensor, y_train_tensor, epochs, learning_rate
)

y_train_pred_nn, y_test_pred_nn = evaluate_nn_model(
    best_model, X_train_tensor, X_test_tensor
)
metrics_nn = calculate_all_metrics(
    y_train_scaled, y_train_pred_nn, y_test_scaled, y_test_pred_nn
)
comparison_metrics[scaler_name]["Neural Network (128-64)"] = metrics_nn

create_regression_comparison_table(
    "Model Comparison - Linear vs Neural Network", comparison_metrics
)

## 3. Visualize the training process

In [ ]:
plot_loss_curves(loss_curves, scalers, architectures)

## Conclusions

- The neural network (128-64) achieved the best performance across all evaluated metrics (MSE, RMSE, MAE, and R²) in both the training and test sets, demonstrating its ability to capture nonlinear relationships in the data.
- Polynomial regression also performed well, with results similar to those of the neural network and outperforming linear regression and ridge regression. This indicates that some of the complexity of the data can be captured without the need for a neural network.
- In terms of the advantages of the neural network, its greater flexibility in modeling nonlinear patterns stands out, which translates into greater predictive accuracy for more complex data sets.
- In terms of disadvantages, we can see that neural networks require more computational resources, longer training times, the need to carefully adjust hyperparameters, and the obligation to correctly normalize the data.
- Linear regression and ridge regression remain useful for simpler relationships, while polynomial regression is appropriate when the data show some nonlinearity. Therefore, the choice between a regression model and a neural network must be carefully evaluated, as in some cases the metrics may be similar and not justify the use of a neural network when a simpler regression model may be sufficient.